# 완성 원화 ZIP → Colab에서 실제 분리·PSD·후속 모델 제작

**완성된 이미지 AI 결과 ZIP만 받습니다.** 외부 AI용 지시서 ZIP을 만들지 않습니다. Live2D PSD2Live, Inochi2D SDK, VRM 본체/액세서리 경로를 구분합니다. Colab 런타임에서 오픈소스 분리·리깅 및 2D 다각도 검수를 수행하며, 24개 이미지 AI 지침 ZIP은 실행하지 않습니다. 실제 앱 동작·표정·물리 품질은 파일 검증과 별개입니다.

In [ ]:
#@title ① 실제 저장소 main 준비
from pathlib import Path
import subprocess,sys,uuid,json,zipfile,shutil,io
REPO=Path('/content/Virtual-pipeline')
REPO_URL='https://github.com/jujumelona/Virtual-pipeline.git'

def _git_checked(command):
    # git exit 128 alone hides the actual cause. Preserve stderr for Colab users.
    try:
        result=subprocess.run(command,capture_output=True,text=True,timeout=300)
    except subprocess.TimeoutExpired as exc:
        raise RuntimeError(f'Git 명령이 300초를 초과했습니다: {command[:5]}') from exc
    if result.returncode:
        diagnostic=(result.stderr or result.stdout or 'Git이 오류 메시지를 출력하지 않았습니다.').strip()
        raise RuntimeError(
            f'Git 명령 실패 (exit={result.returncode}): {command[:5]}\n'
            f'진단 출력:\n{diagnostic[-6000:]}'
        )
    return result

if (REPO/'.git').exists():
    for command in (
        ['git','-C',str(REPO),'fetch','origin','main'],
        ['git','-C',str(REPO),'checkout','-B','main','origin/main'],
    ):
        _git_checked(command)
else:
    # A previous interrupted clone can leave a non-Git directory at REPO.
    # Clone to a fresh staging path first; preserve the old directory as backup.
    staged=REPO.with_name(REPO.name+'-clone-'+uuid.uuid4().hex[:12])
    _git_checked(['git','clone','--depth','1','--branch','main',REPO_URL,str(staged)])
    if REPO.exists() or REPO.is_symlink():
        backup=REPO.with_name(REPO.name+'-backup-'+uuid.uuid4().hex[:12])
        REPO.rename(backup)
        print(f'기존 비-Git 경로는 삭제하지 않고 보관했습니다: {backup}',flush=True)
    staged.rename(REPO)
sys.path.insert(0,str(REPO))
from tools.colab_mode_ui import DISPLAY_TO_MODE,DISPLAY_TO_ASSET,select_mode
print('저장소 준비 완료; ZIP 제작이 아니라 기존 결과 처리입니다.')


In [ ]:
#@title ② 제작 방식/자산 선택
TOP_MODE_LABEL='FACE 공통(구도 없음)' #@param ['FACE 공통(구도 없음)','Live2D 바스트업','Live2D 상반신','Live2D 전신','Inochi2D 상반신','Inochi2D 전신','3D']
ASSET_LABEL='얼굴만' #@param ['얼굴만','신체만','전체 캐릭터','헤어만','액세서리만','의상만']
EDITION='free' #@param ['free','pro']
MODE=DISPLAY_TO_MODE[TOP_MODE_LABEL]
ASSET=DISPLAY_TO_ASSET[ASSET_LABEL]
select_mode(MODE,ASSET)
if ASSET=='face':print('공통 FACE: 바스트업·상반신·전신 구도와 무관한 얼굴·목 원화 1종만 사용합니다.')
if MODE.startswith('live2d') and ASSET=='body':
    print('FREE에서도 신체만 원화 제작/리깅 가능 — 실제 모델은 검증 단계를 별도로 통과해야 합니다')
if MODE.startswith('live2d'):
    print('Live2D 에디션:', EDITION, '— FACE는 얼굴·목 원화만 제작하며 별도 의상과 결합한 뒤 리깅해야 합니다')
if MODE=='3d':
    print('3D 개별 헤어/의상 VRM 생성은 미지원입니다. 전체 캐릭터 또는 액세서리를 선택해야 합니다.')
if MODE=='3d' and ASSET not in ('character','accessory'):
    raise ValueError('3D는 얼굴·헤어·의상까지 모두 그려진 완성 2D 캐릭터 원화 또는 액세서리를 선택합니다.')
if MODE=='3d' and ASSET=='character':
    print('3D 전체 캐릭터: 앞·뒤·좌·우 2D 캐릭터 원화에서 메쉬와 VRM을 제작합니다. FACE 단독 원화는 사용하지 않습니다.')
print(MODE,ASSET,'선택 완료. 다음 셀에는 이미지 AI가 만든 원화·참고 PNG만 담긴 ARTWORK.zip을 업로드하세요. 실제 파츠 분리는 Colab이 담당합니다.')

In [ ]:
#@title ③ 완성된 결과 ZIP 업로드
from google.colab import files
WORK=Path('/content/vts_zip_jobs')/uuid.uuid4().hex
WORK.mkdir(parents=True)
inputs=files.upload()
if len(inputs)!=1:raise ValueError('외부 이미지 AI의 완성 결과 ZIP 정확히 하나가 필요합니다.')
name,bytes_=next(iter(inputs.items()))
if not name.lower().endswith('.zip') or not bytes_:raise ValueError('완성 그림 ZIP이어야 합니다.')
SOURCE=WORK/'finished_artwork.zip';SOURCE.write_bytes(bytes_)
REF_IMG=None;REF_VRM=None
if MODE!='3d' and ASSET not in ('body','face'):
    one=files.upload()
    if len(one)!=1:raise ValueError('같은 캔버스의 FACE 기준 이미지 한 개가 필요합니다.')
    n,b=next(iter(one.items()));e=Path(n).suffix.lower()
    if e not in ('.png','.jpg','.jpeg','.webp'):raise ValueError('FACE 기준 이미지는 PNG/JPG/WebP')
    REF_IMG=WORK/('reference'+e);REF_IMG.write_bytes(b)
if MODE=='3d' and ASSET=='accessory':
    one=files.upload()
    if len(one)!=1:raise ValueError('기존 캐릭터 VRM 한 개가 필요합니다.')
    n,b=next(iter(one.items()))
    if Path(n).suffix.lower()!='.vrm':raise ValueError('VRM을 업로드하세요.')
    REF_VRM=WORK/'base_avatar.vrm';REF_VRM.write_bytes(b)
print('실제 ZIP 업로드됨:',SOURCE.stat().st_size,'bytes')

In [ ]:
#@title ④ ZIP/파츠 실제 바이트 검증 (모델 설치 전에)
from zipfile import ZipFile
from pathlib import PurePosixPath
from PIL import Image
from tools.vts_segmentation_profiles import (
    get_segmentation_profile, select_source_segmentation,
    write_segmentation_plan,
)
SEGMENTATION_PLAN=write_segmentation_plan(
    MODE, ASSET, WORK/'segmentation_plan.json')
# A 3D reference image is not a See-through PSD. Independent 3D hair/outfit
# are not implemented and must fail here, before installing any GPU models.
select_source_segmentation(MODE, ASSET)
print('Colab 담당 분리 경로:', SEGMENTATION_PLAN['route'])
with ZipFile(SOURCE) as z:
    entries=[x for x in z.infolist() if not x.is_dir()]
    if not entries or len(entries)>600 or sum(x.file_size for x in entries)>1024**3 or z.testzip() is not None:
        raise ValueError('결과 ZIP이 비어 있거나 손상되었거나 과도합니다.')
    names=[x.filename for x in entries]
    if len(names)!=len(set(names)):raise ValueError('중복 ZIP 파일')
    for x in entries:
        p=PurePosixPath(x.filename)
        if x.filename.startswith('/') or '..' in p.parts or '\\' in x.filename or ((x.external_attr>>16)&0o170000)==0o120000:
            raise ValueError('ZIP 경로가 안전하지 않습니다: '+x.filename)
if MODE!='3d':
    import importlib
    import tools.vts_ai_pack_import as ai_pack_import
    importlib.reload(ai_pack_import)  # force current code after rerunning setup cell
    from tools.vts_source_only_pack import is_source_only_zip, prepare_source_only_2d
    SOURCE_ONLY=is_source_only_zip(SOURCE)
    if SOURCE_ONLY:
        # Image AI may supply one-by-one part_art/*.png artworks; their original
        # alpha and hidden neck/ear regions must be preserved through step ⑤.
        SOURCE_QA=prepare_source_only_2d(SOURCE, WORK/'source_preflight', asset=ASSET)
        inspect={'canvas': tuple(SOURCE_QA['canvas']), 'parts': []}
        print('원화 전용 ZIP 검사:', SOURCE_QA['state'], '— 이미지 AI에는 파츠 PNG 또는 PSD를 요구하지 않습니다.')
    else:
        inspect=ai_pack_import.inspect_large_ai_pack(
            SOURCE,edition=EDITION if MODE.startswith('live2d') else 'pro')
        from tools.vts_multiview_runtime import prepare_multiview_input
        SOURCE_QA=prepare_multiview_input(
            SOURCE, WORK/'source_preflight', asset=ASSET,
            edition=EDITION if MODE.startswith('live2d') else 'pro',
            require_views=True)
        print('다각도/파츠 원화 사전 검사:', SOURCE_QA['parts']['status'])
    if REF_IMG:
        with Image.open(REF_IMG) as image:
            if image.size!=inspect['canvas']:raise ValueError('FACE와 파츠 캔버스 크기 불일치')
    if MODE.startswith('live2d') and ASSET=='body' and not SOURCE_ONLY:
        # SOURCE_QA already checked both visible layers and hidden underpaint.
        # Never run the legacy visible-only neck/torso check a second time.
        skin_joint=SOURCE_QA['parts']['neck']
        if SOURCE_QA['parts']['status']!='SEMANTIC_SOURCE_PREFLIGHT_PASS':
            raise RuntimeError('목/몸통 연결 사전 검사 미통과')
        print('목/몸통 원화 회전 연결:',skin_joint)
    if SOURCE_ONLY:
        print('그림 ZIP 원본 사전 검사:', SOURCE_QA['master'],
              '작성된 독립 파츠:', SOURCE_QA.get('authored_parts_present', False))
    else:
        print('독립 원화 파츠:',len(inspect['parts']),'리깅 전 원화 상태입니다.')
else:
    from tools.vts_multiview_runtime import prepare_3d_turnaround_input
    SOURCE_QA=prepare_3d_turnaround_input(
        SOURCE, WORK/'turnaround_preflight', asset=ASSET)
    views={}
    for f in names:
        p=PurePosixPath(f)
        if len(p.parts) in (2,3) and p.parts[-2]=='views' and f.endswith('.png'):
            for side in ('front','back','left','right'):
                if p.stem.lower().endswith('_'+side):
                    if side in views:raise ValueError('중복 시점: '+side)
                    views[side]=f
    if set(views)!=set(('front','back','left','right')):raise ValueError('3D ZIP에 독립 4면도 front/back/left/right가 모두 필요합니다.')
    import hashlib
    with ZipFile(SOURCE) as z:
        data={side:z.read(f) for side,f in views.items()}
    if len({hashlib.sha256(b).digest() for b in data.values()})!=4:raise ValueError('똑같은 그림을 4개 시점으로 중복 제출했습니다.')
    for side,b in data.items():
        with Image.open(io.BytesIO(b)) as im:
            if im.format!='PNG' or min(im.size)<256:raise ValueError('잘못된 3D 시점 그림: '+side)
    if MODE=='3d' and ASSET=='accessory':
        from tools.vts_finished_zip_routing import resolve_accessory_placement
        ATTACHMENT=resolve_accessory_placement(SOURCE,views['front'])
        print('원본 ZIP에서 확인된 액세서리 부착 위치:',ATTACHMENT)
    print('3D 4면도 PNG 검증됨. 실제 시각적 각도 일치 검증은 별개입니다.')

In [ ]:
#@title ⑤ 필요한 엔진만 준비하고 실제 네이티브 제작
from zipfile import ZIP_DEFLATED
import subprocess
OUT=WORK/'produced';OUT.mkdir()
RIGGED=False
NATIVE=None
NATIVE_KIND=None
SOURCE_PSD=None
EXTRA={'segmentation_plan': SEGMENTATION_PLAN}
if ASSET=='face':
    # FACE receives a genuinely independent head-and-neck illustration.
    # A face-only artwork is NOT a complete avatar rig or full-body PSD.
    from zipfile import ZipFile
    FACE_ONLY_SOURCE=OUT/'face_source_only.zip'
    with ZipFile(FACE_ONLY_SOURCE,'w',ZIP_DEFLATED) as packed:
        packed.write(SOURCE,'input_reference/face_artwork.zip')
        packed.writestr('README_FACE.txt',
            'FACE/neck source only. Merge with a registered clothed OUTFIT, '
            'then perform eye/mouth segmentation and rigging downstream. '
            'No MOC3, VRM or native rig was generated.')
    NATIVE=FACE_ONLY_SOURCE
    NATIVE_KIND=None
    SOURCE_PSD=None
    RIGGED=False
    EXTRA['stage']='FACE_SOURCE_ONLY_NOT_RIGGED'
elif MODE!='3d':
    subprocess.run([sys.executable,'-m','pip','install','-q','psd-tools==1.14.2','Pillow','numpy','scipy>=1.13','scikit-image==0.26.0'],check=True,timeout=1200)
    imported=None
    if SOURCE_ONLY and ASSET=='body' and not SOURCE_QA.get('authored_parts_present'):
        from tools.vts_setup import prepare as prepare_see_through
        import os
        ready=prepare_see_through(qwen=False,install=True,sam_body=ASSET=='body')
        setup=json.loads(Path(ready).read_text(encoding='utf-8'))
        os.environ['VTUBER_SEETHROUGH_PYTHON']=setup['see_through_python']
        print('실제 파츠 원화가 없으므로 See-through 분리기를 준비합니다.')
    if not MODE.startswith('live2d'):
        if SOURCE_ONLY:
            if SOURCE_QA.get('authored_parts_present'):
                from tools.vts_authored_part_art import assemble_authored_parts
                report=assemble_authored_parts(
                    SOURCE,Path(SOURCE_QA['master']),OUT/'inochi_authored_parts',
                    asset=ASSET,edition='pro',
                    scope='full' if MODE=='inochi2d_full' else 'upper',
                    enforce_required_parts=True)
                imported={'master':Path(SOURCE_QA['master']),'psd':Path(report['psd'])}
                EXTRA['authored_part_art_neck_geometry']=report['neck_source_overlap']
                print('Inochi2D: 실제 파츠 원화 PSD 조립 및 숨은 목 픽셀 확인됨')
            else:
                if ASSET=='body':
                    from tools.vts_production import run_see_through
                    psd=run_see_through(Path(SOURCE_QA['master']),OUT/'decomposition',
                                        third_party=Path('/content/vtuber_builder/third_party/see-through'),
                                        qwen=False)
                else:
                    from tools.vts_asset_input import prepare_detached_asset
                    psd, mask_report = prepare_detached_asset(
                        Path(SOURCE_QA['master']),OUT/'detached_alpha',
                        asset_kind=ASSET,qwen=False,layer_count=4,pass_budget=0,
                        third_party=Path('/content/vtuber_builder/third_party'))
                    EXTRA['detached_alpha_report']=mask_report
                    EXTRA['additional_micro_split_required']=True
                imported={'master':Path(SOURCE_QA['master']),'psd':Path(psd)}
        else:
            from tools.vts_ai_pack_import import prepare_large_ai_pack
            imported=prepare_large_ai_pack(SOURCE,OUT/'imported',edition=EDITION)
        SOURCE_PSD=Path(imported['psd'])
    if MODE.startswith('live2d'):
        # AI ZIP creates artwork. The Colab runtime executes the OSS backends.
        # Preserve authored parts; do not force another See-through split.
        if ASSET=='body':
            from tools.vts_psd2live import prepare_psd2live
            prepare_psd2live()
        from tools.vts_multiview_runtime import process_completed_ai_2d_zip
        LIVE2D_PIPELINE=process_completed_ai_2d_zip(
            SOURCE, OUT/'live2d_pipeline', edition=EDITION,
            scope='upper' if MODE in ('live2d_bust_up','live2d_upper') else 'full',
            asset=ASSET, reference_image=REF_IMG, require_views=True,
            mode=MODE)
        result=LIVE2D_PIPELINE['native']
        SOURCE_PSD=Path(LIVE2D_PIPELINE['source_artwork'])
        NATIVE=Path(result['package'])
        RIGGED=ASSET=='body' and result.get('state')=='psd2live_rigged_unverified' and result.get('moc3_generated') is True
        if ASSET=='body' and not RIGGED:
            raise RuntimeError('PSD2Live가 네이티브 모델을 만들지 못했습니다.')
        NATIVE_KIND='moc3' if RIGGED else None
        EXTRA['quality_status']='native_artifact_created_runtime_and_visual_review_required'
        if SOURCE_ONLY and ASSET!='body':
            EXTRA['fine_asset_segmentation_state']='DETACHED_ALPHA_SINGLE_LAYER_REFINEMENT_REQUIRED'
            print('독립 자산: Colab에서 전경 알파/PSD를 생성했지만 움직임 단위 세분화는 미완료입니다. 생성 AI에 파츠 제작을 다시 요구하지 않습니다.')
        EXTRA['independent_pose_qa']=LIVE2D_PIPELINE.get('pose_qa_error') or (
            (LIVE2D_PIPELINE.get('pose_qa') or {}).get('source_render_checks_passed'))
    elif ASSET=='body':
        from tools.colab_mode_prepare import prepare_selected_mode
        prepare_selected_mode('inochi2d',provided_2d_layers=True)
        if SOURCE_ONLY:
            from tools.vts_production import psd_to_registered_rgba
            _,rgba,_=psd_to_registered_rgba(
                SOURCE_PSD,OUT/'registered_see_through',artmesh_max=None)
        else:
            rgba=OUT/'registered_layers.zip'
            with ZipFile(rgba,'w',ZIP_DEFLATED) as z:
                for part in inspect['parts']:
                    canvas=Image.new('RGBA',inspect['canvas'],(0,0,0,0))
                    canvas.paste(part['image'],(part['left'],part['top']))
                    buf=io.BytesIO();canvas.save(buf,'PNG')
                    z.writestr(part['id']+'.png',buf.getvalue())
        from vtuber_pipeline.common.schemas import SourceSet
        from vtuber_pipeline.two_d.build import build_inochi2d
        from vtuber_pipeline.two_d.inochi_bridge import inspect_native_inp
        result=build_inochi2d(SourceSet('inochi2d',str(imported['master']),
            user_layers_zip=str(rgba),output_dir=str(OUT/'inochi'),
            artwork_profile='vts_auto'))
        if result.status!='complete' or not result.primary_file:raise RuntimeError('Inochi SDK 내보내기 실패: '+str(result.error))
        NATIVE=Path(result.primary_file);inspect_native_inp(str(NATIVE))
        RIGGED=True;NATIVE_KIND='inp'
        EXTRA['quality_status']='Inochi native file exists; side/neck animation and live app motion not verified'
        EXTRA['native_side_profile_keyforms_bound']=False
    else:
        NATIVE=SOURCE_PSD
else:
    from tools.colab_mode_prepare import prepare_selected_mode
    prepare_selected_mode('3d',prewarm_first_gpu=False)
    from tools.colab_generation_process import run_isolated
    from tools.colab_native import _event_printer
    paths={}
    with ZipFile(SOURCE) as z:
        for side,f in views.items():
            p=OUT/(side+'.png');p.write_bytes(z.read(f));paths[side]=p
        face_candidates=[n for n in names if PurePosixPath(n).parts[-2:]==('details','face.png')]
        if len(face_candidates)>1:raise ValueError('중복 얼굴 참조 이미지')
        face=None
        if face_candidates:
            face=OUT/'face.png';face.write_bytes(z.read(face_candidates[0]))
    if ASSET=='character':
        args=[str(paths['front']),'corporation',None,str(face) if face else None,
              str(paths['back']),True,2048,str(paths['left']),str(paths['right']),'canonical']
        mode='avatar';EXTRA={'used_views':list(views.values())}
    else:
        if not REF_VRM:raise ValueError('기존 VRM 없이 독립 액세서리 완성 불가')
        args=[False,str(REF_VRM),None,str(paths['front']),ATTACHMENT,'',0.,0.,0.,1.]
        mode='accessory';EXTRA={'used_views':[views['front']],
                               'unused_views':[views[x] for x in ('back','left','right')]}
    result=run_isolated(mode,args,on_event=_event_printer)
    if len(result)<3 or not str(result[0]).startswith('✅') or not result[2]:raise RuntimeError('3D VRM 제작 실패: '+str(result[:2]))
    NATIVE=Path(result[2])
    from vtuber_pipeline.avatar.validator import validate_vrm
    checked=validate_vrm(str(NATIVE),str(OUT/'validation'),product_contract=True)
    if not checked.get('passed'):raise RuntimeError('3D VRM 검증 실패: '+str(checked.get('error') or checked['checks'])[:550])
    RIGGED=True;NATIVE_KIND='vrm'
if not NATIVE or not NATIVE.is_file():raise RuntimeError('최종 결과 파일이 생성되지 않았습니다.')
print('실제 제작 단계 종료, 네이티브 파일:',NATIVE_KIND or '없음 (교체용 PSD 원화)')

In [ ]:
#@title ⑥ 실제 산출 파일과 원본 ZIP 포장·다운로드
from google.colab import files
import json,shutil
package=OUT/'FINAL_VTUBER_DELIVERY.zip'
if NATIVE.suffix.lower()=='.zip':
    shutil.copyfile(NATIVE,package)
else:
    with ZipFile(package,'w',ZIP_DEFLATED) as z:z.write(NATIVE,('native/' if RIGGED else 'artwork/')+NATIVE.name)
with ZipFile(package,'a',ZIP_DEFLATED) as z:
    if 'input_reference/finished_ai_artwork.zip' in z.namelist():raise RuntimeError('중복 원본 결과 파일')
    z.write(SOURCE,'input_reference/finished_ai_artwork.zip')
    if SOURCE_PSD:z.write(SOURCE_PSD,'source_artwork/imported_parts.psd')
    if REF_IMG:z.write(REF_IMG,'input_reference/face_reference'+REF_IMG.suffix)
    if REF_VRM:z.write(REF_VRM,'input_reference/base_avatar.vrm')
    # Store actual evidence, not a fake visual-quality PASS.
    if MODE.startswith('live2d') and 'LIVE2D_PIPELINE' in globals():
        root=OUT/'live2d_pipeline'
        for artifact, arc in (
            (root/'pipeline_result.json','qa/pipeline_result.json'),
            (root/'source_qa/artwork_preflight.json','qa/artwork_preflight.json'),
            (root/'source_qa/reference_views/multiview_manifest.json','qa/multiview_manifest.json'),
            (root/'quality/quality_verdict.json','qa/quality_verdict.json'),
            (root/'quality/multiview_pose_reference_comparison.png','qa/multiview_pose_reference_comparison.png'),
            (root/'quality/multiview_vs_native_moc3_comparison.png','qa/multiview_vs_native_moc3_comparison.png'),
            (root/'independent_pose/pose_review.json','qa/independent_pose_report.json'),
            (root/'native_moc3_pose/native_pose_report.json','qa/native_moc3_pose_report.json')):
            if artifact.is_file():z.write(artifact,arc)
        for frame in sorted((root/'independent_pose').glob('*.png')):
            z.write(frame,'qa/secondary_rig_frames/'+frame.name)
        for frame in sorted((root/'native_moc3_pose').glob('*.png')):
            z.write(frame,'qa/native_moc3_frames/'+frame.name)
        for alternative in sorted((root/'independent_pose/independent_nijilive').glob('*.inp')):
            z.write(alternative,'alternatives/nijilive/'+alternative.name)
    elif MODE=='3d' and (WORK/'turnaround_preflight/turnaround_report.json').is_file():
        z.write(WORK/'turnaround_preflight/turnaround_report.json','qa/turnaround_report.json')
    z.writestr('metadata/production_status.json',json.dumps(
        {'mode':MODE,'asset':ASSET,'native_format':NATIVE_KIND,
         'native_files_created':RIGGED,'application_runtime_verified':False,
         'visual_artwork_quality_verified':False,
         'native_pose_rendered':bool(globals().get('LIVE2D_PIPELINE',{}).get('native_pose',{} ) and globals().get('LIVE2D_PIPELINE',{}).get('native_pose',{}).get('native_pose_rendered')),
         'native_pose_activity_passed':bool(globals().get('LIVE2D_PIPELINE',{}).get('native_pose',{} ) and globals().get('LIVE2D_PIPELINE',{}).get('native_pose',{}).get('pose_activity_passed')),
         'multiview_reference_applied_to_native_keyforms':False,
         'manual_attachment_needed':not RIGGED,
         'face_only_source_no_native_rig':ASSET=='face','3d_views':EXTRA},ensure_ascii=False,indent=2))
    z.writestr('README_RESULT.md','원화 품질과 네이티브 모델의 자세별 품질은 별도 검증해야 합니다. qa/에 독립 리그의 시점별 보조 렌더가 있더라도 Cubism/VTube Studio 네이티브 MOC3 검증이 아닙니다.\n')
with ZipFile(package) as z:
    if z.testzip() is not None:raise RuntimeError('결과 ZIP 손상')
    if RIGGED and NATIVE_KIND=='moc3' and not (any(n.endswith('.moc3') for n in z.namelist()) and any(n.endswith('.model3.json') for n in z.namelist())):raise RuntimeError('네이티브 MOC3 누락')
    if RIGGED and NATIVE_KIND in ('vrm','inp') and not any(n.endswith('.'+NATIVE_KIND) for n in z.namelist()):raise RuntimeError('네이티브 모델 누락')
print('결과:',package,'실제 네이티브:',RIGGED,'실물 구동 품질: 미검증')
files.download(str(package))